# BTC Direction Classifier — End-to-End Model

**What this notebook does:**  
We build two machine learning models that try to predict whether Bitcoin's price will go **UP**, **DOWN**, or **stay flat** the next day.  
We compare how well each one works — both as a predictor and as a trading strategy.

---

**Think of it like this:**  
Imagine you are a weather forecaster. Every morning you look at humidity, wind speed, temperature, etc. — and predict whether it will rain tomorrow.  
We are doing the same thing, but instead of rain, we are predicting whether BTC will go up or down.  
We then test two different forecasting methods and compare which one is better.

---

**Table of Contents**
1. Install & Import Libraries
2. Load the Data
3. Re-label the Data (Define BUY / HOLD / SELL)
4. Explore the Data
5. Prepare Features & Target
6. Walk-Forward Cross-Validation
7. Model A — LightGBM Baseline
8. Hyperparameter Tuning — LightGBM
9. LightGBM Production Backtest
10. LightGBM Results Summary
11. Model B — Logistic Regression (Parametric)
12. Hyperparameter Tuning — Logistic Regression
13. Production Backtest — Both Models
14. Head-to-Head Comparison & Final Summary


---
## 1. Install & Import Libraries

Libraries are like pre-built toolboxes that other people have created.  
Instead of writing everything from scratch, we just import them.

| Library | What it does |
|---|---|
| `pandas` | Reads and manages table data (like Excel) |
| `numpy` | Does fast math on large lists of numbers |
| `lightgbm` | The machine learning model we use |
| `optuna` | Automatically finds the best settings for our model |
| `sklearn` | Tools for splitting data, measuring accuracy, etc. |
| `matplotlib` | Draws charts and graphs |

In [ ]:
# Install required libraries (run this once)
# The exclamation mark means: run this as a terminal command, not Python
!pip install lightgbm optuna scikit-learn pandas numpy matplotlib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress noisy logs

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, classification_report

# ── Global settings ─────────────────────────────────────────────────────────
RANDOM_STATE     = 42          # Makes results reproducible (same random seed every run)
N_SPLITS         = 5           # How many train/test windows we use
SIGMA_THRESHOLD  = 1.0         # How big a move must be to count as BUY or SELL
PROBA_THRESHOLD  = 0.45        # Minimum confidence before we act on a signal
TRANSACTION_COST = 0.001       # 10 basis points (0.1%) per trade — exchange fee
CLASS_NAMES      = ['BUY', 'HOLD', 'SELL']

print('All libraries loaded successfully!')

---
## 2. Load the Data

Our dataset has **one row per day**, going from July 2018 to May 2026.  
Each row contains a set of **features** — these are measurements that describe what the market looked like on that day.  

Think of each row as a "patient file" in a hospital — it records all the vital signs for that day.

In [ ]:
# Load the CSV file into a pandas DataFrame (a table)
df = pd.read_csv('btc_features_labeled_vol_threshold.csv')

# Convert the 'date' column from plain text to an actual date format Python understands
df['date'] = pd.to_datetime(df['date'])

# Sort by date — very important for time-series, oldest row first
df = df.sort_values('date').reset_index(drop=True)

print(f'Dataset shape : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Date range    : {df["date"].min().date()}  →  {df["date"].max().date()}')
print(f'\nFirst 3 rows:')
df.head(3)

In [ ]:
# What features (columns) do we have?
# Let's explain what each one means

feature_descriptions = {
    'log_return'        : "Today's BTC return (log scale) — did price go up or down today?",
    'lag_return_1'      : "Yesterday's return",
    'lag_return_2'      : "Return from 2 days ago",
    'lag_return_3'      : "Return from 3 days ago",
    'return_7d'         : "Total return over the past 7 days",
    'return_14d'        : "Total return over the past 14 days",
    'return_30d'        : "Total return over the past 30 days",
    'dist_ma20'         : "How far price is from its 20-day average (trend indicator)",
    'dist_ma50'         : "How far price is from its 50-day average",
    'dist_ma200'        : "How far price is from its 200-day average (long-term trend)",
    'upper_wick'        : "Candle upper wick size — how much price tried to go up but failed",
    'lower_wick'        : "Candle lower wick size — how much price tried to go down but recovered",
    'range_norm'        : "Daily price range (high minus low), normalised",
    'realized_vol_14'   : "How much BTC has been moving around over the past 14 days (volatility)",
    'vol_ratio_7_30'    : "Short-term volatility vs long-term — is market getting more or less jumpy?",
    'atr_pct'           : "Average True Range — another volatility measure",
    'rsi_14'            : "RSI indicator — is BTC overbought (>70) or oversold (<30)?",
    'volume_ratio_20'   : "Today's trading volume vs its 20-day average — is there unusual activity?",
    'sp500_ret_ma7'     : "S&P 500 recent trend — is the broader stock market going up or down?",
    'nasdaq_ret_ma7'    : "Nasdaq recent trend — tech stocks performance",
    'vix_level_ma7'     : "VIX (Fear Index) — high VIX means markets are scared",
    'fear_greed_ma7'    : "Crypto Fear & Greed Index — sentiment of crypto market",
    'fear_greed_change' : "How much sentiment changed recently",
    'bb_width_20'       : "Bollinger Band width — how wide the price channel is (volatility)",
    'bb_position_20'    : "Where price sits within its Bollinger Bands (0=bottom, 1=top)",
    'macd_hist'         : "MACD histogram — momentum indicator (positive=bullish)",
    'roc_10'            : "Rate of Change over 10 days — how fast price is moving",
    'dxy_ret_ma7'       : "US Dollar Index trend — strong dollar often hurts BTC",
}

print(f'We have {len(feature_descriptions)} features (inputs) for our model:\n')
for col, desc in feature_descriptions.items():
    print(f'  {col:<22} →  {desc}')

---
## 3. Re-label the Data — Define BUY / HOLD / SELL

Before we can train the model, we need to tell it **what the right answer was** for each day.

**The key decision: how big does a move need to be to call it a BUY or SELL?**

We use **1.0 sigma (σ)** — meaning we only label a day as BUY or SELL if the next day's move is at least **1 full standard deviation** in size. Smaller moves get labeled HOLD.

**Why 1.0σ and not something smaller?**  
Think of it like grading student essays:
- If you give an A to almost every essay (low bar = 0.5σ), your grades become meaningless — the model can't learn what "really good" looks like
- If you only give an A to truly outstanding work (high bar = 1.0σ), there is a clear, learnable difference between A and B

We tested this — at 0.5σ the model's log-loss was **1.16** (worse than random guessing!). At 1.0σ it dropped to **0.83**. The cleaner labels make a huge difference.

In [ ]:
# Step 1: Calculate daily volatility
# 'realized_vol_7' is annualised (like a yearly rate). We convert it to a daily number.
# There are ~252 trading days per year, so we divide by sqrt(252)
df['daily_vol'] = df['realized_vol_7'] / np.sqrt(252)

# Step 2: Create the forward return — what actually happened NEXT day
# .shift(-1) moves the column up by 1 row (tomorrow's return becomes today's label)
df['fwd_return'] = df['log_return'].shift(-1)

# Step 3: Calculate the threshold — 1 standard deviation of daily movement
threshold = df['daily_vol'] * SIGMA_THRESHOLD   # SIGMA_THRESHOLD = 1.0

# Step 4: Apply the labels
# Start by calling every day HOLD
df['label'] = 'HOLD'
# Upgrade to BUY if next day moved UP by more than 1σ
df.loc[df['fwd_return'] >  threshold, 'label'] = 'BUY'
# Upgrade to SELL if next day moved DOWN by more than 1σ
df.loc[df['fwd_return'] < -threshold, 'label'] = 'SELL'

# Step 5: Remove the last row — it has no forward return (there is no "tomorrow")
df = df.dropna(subset=['fwd_return']).reset_index(drop=True)

# Show the result
label_counts = df['label'].value_counts()
print('Label distribution after 1.0σ threshold:')
print()
for label, count in label_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<5}  {count:>4} days ({pct:.1f}%)  {bar}')

print(f'\nTotal days: {len(df)}')
print(f'\nInterpretation: Only {(label_counts["BUY"]+label_counts["SELL"])/len(df)*100:.0f}% of days'
      f' get a directional label — these are the genuinely large moves.')

---
## 4. Explore the Data

Before building the model, let's look at the data visually to understand what we're working with.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Data Exploration', fontsize=14, fontweight='bold', y=1.01)

# Chart 1: BTC price over time
ax = axes[0, 0]
btc_price = np.exp(df['log_return'].cumsum())  # Reconstruct price index from log returns
ax.plot(df['date'], btc_price, color='#185FA5', linewidth=1)
ax.set_title('BTC Price Index (reconstructed)', fontsize=11)
ax.set_xlabel('Date'); ax.set_ylabel('Price index')
ax.tick_params(labelsize=8)

# Chart 2: Label distribution over time (colour-coded)
ax = axes[0, 1]
colors = {'BUY': '#639922', 'HOLD': '#888780', 'SELL': '#A32D2D'}
for label in ['BUY', 'HOLD', 'SELL']:
    mask = df['label'] == label
    ax.scatter(df.loc[mask, 'date'], df.loc[mask, 'fwd_return'],
               c=colors[label], s=3, alpha=0.5, label=label)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_title('Labels vs Actual Next-Day Return', fontsize=11)
ax.set_xlabel('Date'); ax.set_ylabel('Next day return')
ax.legend(fontsize=8, markerscale=3)
ax.tick_params(labelsize=8)

# Chart 3: Distribution of next-day returns with ±1σ lines
ax = axes[1, 0]
ax.hist(df['fwd_return'], bins=80, color='#185FA5', alpha=0.7, edgecolor='none')
avg_thresh = threshold.mean()
ax.axvline( avg_thresh, color='#639922', linewidth=2, linestyle='--', label=f'+1σ ({avg_thresh:.3f})')
ax.axvline(-avg_thresh, color='#A32D2D', linewidth=2, linestyle='--', label=f'-1σ ({-avg_thresh:.3f})')
ax.set_title('Distribution of Next-Day Returns', fontsize=11)
ax.set_xlabel('Log return'); ax.set_ylabel('Count')
ax.legend(fontsize=8)
ax.tick_params(labelsize=8)

# Chart 4: Feature correlations with forward return
ax = axes[1, 1]
excl = ['date', 'label', 'fwd_return', 'daily_vol', 'realized_vol_7']
feat_cols = [c for c in df.columns if c not in excl]
corrs = df[feat_cols].corrwith(df['fwd_return']).sort_values()
colors_bar = ['#A32D2D' if c < 0 else '#639922' for c in corrs]
ax.barh(corrs.index, corrs.values, color=colors_bar, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Feature Correlation with Next-Day Return', fontsize=11)
ax.set_xlabel('Pearson correlation')
ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('data_exploration.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as data_exploration.png')

---
## 5. Prepare Features & Target

**Features (X)** = the inputs to the model (the 28 columns that describe each day)  
**Target (y)** = what we want the model to predict (BUY / HOLD / SELL)

In [ ]:
# Columns we do NOT use as inputs:
# - 'date'          → not a number, just the calendar date
# - 'label'         → this IS what we're predicting, can't be an input
# - 'fwd_return'    → this is the future return — the model can't know this in advance!
# - 'daily_vol'     → we created this ourselves as a helper, it leaks into the label
# - 'realized_vol_7'→ already used to compute daily_vol, keep it in features via vol_ratio
exclude_cols = ['date', 'label', 'fwd_return', 'daily_vol', 'realized_vol_7']

feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].values   # 2D array: rows = days, columns = features
y = df['label'].values         # 1D array: BUY / HOLD / SELL for each day
fwd_returns = df['fwd_return'].values  # We use this later for backtesting
dates = df['date'].values

print(f'X shape: {X.shape}  →  {X.shape[0]} days, {X.shape[1]} features')
print(f'y shape: {y.shape}  →  one label per day')
print(f'\nFeature columns ({len(feature_cols)} total):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

---
## 6. Walk-Forward Cross-Validation

**Why we can't use random train/test splits with time-series data:**

Normally in machine learning, you randomly pick 80% of your data to train on and test on the remaining 20%. But with financial data, that would be **cheating** — you'd be training on data from 2024 and testing on data from 2020. The model would already "know" the future.

**Walk-forward validation** fixes this. It works like this:

```
Fold 1:  [TRAIN: 2018–2021]  →  [TEST: 2021]
Fold 2:  [TRAIN: 2018–2022]  →  [TEST: 2022]
Fold 3:  [TRAIN: 2018–2023]  →  [TEST: 2023]
Fold 4:  [TRAIN: 2018–2024]  →  [TEST: 2024]
Fold 5:  [TRAIN: 2018–2025]  →  [TEST: 2025–2026]
```

The model always trains on the **past** and tests on the **future**. This is how it would work in real life.

In [ ]:
# TimeSeriesSplit creates the walk-forward windows automatically
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
splits = list(tscv.split(X))

print('Walk-forward fold structure:')
print()
for fold, (train_idx, test_idx) in enumerate(splits, 1):
    train_start = df.iloc[train_idx[0]]['date'].date()
    train_end   = df.iloc[train_idx[-1]]['date'].date()
    test_start  = df.iloc[test_idx[0]]['date'].date()
    test_end    = df.iloc[test_idx[-1]]['date'].date()
    print(f'  Fold {fold}:  Train [{train_start} → {train_end}] ({len(train_idx)} days)'
          f'  |  Test [{test_start} → {test_end}] ({len(test_idx)} days)')

---
## 7. Baseline Model

### What is LightGBM?

LightGBM (Light Gradient Boosting Machine) is a type of model that works by building many small **decision trees** and combining them.

Imagine you are trying to decide whether to go out without an umbrella:
- Tree 1 asks: "Is it cloudy?" → If yes, lean towards rain
- Tree 2 asks: "Is the humidity high?" → If yes, lean even more towards rain
- Tree 3 asks: "Did it rain yesterday?" → If yes, lean even more
- ... and so on for 200 trees

Each tree fixes the mistakes of the previous ones. The final answer is a vote across all 200 trees.

### What is Log-Loss?

Log-loss measures **how confident and correct** the model is. Lower is better.
- A model that randomly guesses scores **1.10** (ln(3) — because there are 3 equal classes)
- Our baseline scores **~0.83** — meaningfully better than guessing
- Our tuned model scores **~0.79** — even better

In [ ]:
# Baseline model hyperparameters — these are the "settings" of the model
# Think of them like the settings on a camera: aperture, shutter speed, ISO
BASELINE_PARAMS = dict(
    objective        = 'multiclass',  # We are predicting 3 classes (BUY/HOLD/SELL)
    num_class        = 3,
    class_weight     = 'balanced',    # Compensate for HOLD being more common than BUY/SELL
    n_estimators     = 200,           # Number of trees to build
    learning_rate    = 0.05,          # How much each tree corrects the previous one
    max_depth        = 6,             # Maximum depth of each tree
    num_leaves       = 31,            # Maximum number of leaf nodes per tree
    subsample        = 0.8,           # Use 80% of rows for each tree (prevents overfitting)
    colsample_bytree = 0.8,           # Use 80% of features for each tree
    random_state     = RANDOM_STATE,
    verbose          = -1,            # Suppress output during training
)

print('Baseline model parameters:')
for k, v in BASELINE_PARAMS.items():
    print(f'  {k:<20} = {v}')

In [ ]:
# Train baseline model on each fold and record log-loss
baseline_ll_per_fold = []

print('Training baseline model across all folds...')
print()
print(f'  {"Fold":<6} {"Test period":<26} {"Log-loss":>10}  {"vs Random":>10}')
print('  ' + '-' * 58)

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    d0 = df.iloc[test_idx[0]]['date'].date()
    d1 = df.iloc[test_idx[-1]]['date'].date()

    # Train the model
    model = lgb.LGBMClassifier(**BASELINE_PARAMS)
    model.fit(X_train, y_train)

    # Predict probabilities on the test set
    # For each day, the model outputs 3 numbers that sum to 1:
    # e.g. [BUY=0.62, HOLD=0.25, SELL=0.13] → model thinks 62% chance of going up
    proba = model.predict_proba(X_test)

    # Calculate log-loss
    ll = log_loss(y_test, proba, labels=CLASS_NAMES)
    baseline_ll_per_fold.append(ll)

    vs_random = np.log(3) - ll   # How much better than random guessing?
    print(f'  {fold:<6} {str(d0)+" → "+str(d1):<26} {ll:>10.4f}  {vs_random:>+10.4f}')

avg_baseline_ll = np.mean(baseline_ll_per_fold)
print('  ' + '-' * 58)
print(f'  {"AVG":<33} {avg_baseline_ll:>10.4f}  {np.log(3)-avg_baseline_ll:>+10.4f}')
print(f'\n  Random baseline would score: {np.log(3):.4f}')
print(f'  Our baseline scores       : {avg_baseline_ll:.4f}  ✅ Better than random!')

In [ ]:
def simulate_trading(signals, forward_returns, cost=TRANSACTION_COST):
    """
    Simulate a simple trading strategy based on model signals.

    Parameters
    ----------
    signals        : list of 'BUY', 'HOLD', or 'SELL' for each day
    forward_returns: actual next-day returns (what actually happened)
    cost           : transaction cost paid when entering/changing position

    Returns
    -------
    Dictionary with performance metrics
    """
    equity   = [1.0]        # Start with £1 (or $1 — normalised)
    position = None          # Current position: None, 'BUY', or 'SELL'
    daily_returns = []
    n_trades  = 0

    for signal, ret in zip(signals, forward_returns):
        prev_equity = equity[-1]

        if signal == 'HOLD':
            # No position — we sit out and earn nothing
            equity.append(prev_equity)
            daily_returns.append(0.0)
            continue

        # If the signal changed, we switch positions and pay the fee
        if signal != position:
            prev_equity *= (1 - cost)   # Pay transaction cost
            position     = signal
            n_trades    += 1

        # Calculate today's P&L based on the actual return
        pnl = ret if signal == 'BUY' else -ret   # SELL means we profit when price falls
        new_equity = prev_equity * (1 + pnl)
        equity.append(new_equity)
        daily_returns.append(new_equity / prev_equity - 1)

    equity        = np.array(equity[1:])    # Remove the initial £1
    daily_returns = np.array(daily_returns)

    # Performance metrics
    total_return = equity[-1] / equity[0] - 1
    sharpe       = daily_returns.mean() / (daily_returns.std() + 1e-9) * np.sqrt(252)
    running_max  = np.maximum.accumulate(equity)
    max_drawdown = ((equity - running_max) / running_max).min()   # Worst peak-to-trough drop
    win_rate     = (daily_returns > 0).mean()

    return {
        'total_return': total_return,
        'sharpe'      : sharpe,
        'max_drawdown': max_drawdown,
        'win_rate'    : win_rate,
        'n_trades'    : n_trades,
        'equity'      : equity,
        'daily_rets'  : daily_returns,
    }

print('simulate_trading() function defined — ready for backtest.')

---
## 7b. Three LGBM Variants — Hyperparameter Comparison

Instead of a single LightGBM model, we now train **three versions** with deliberately different hyperparameter philosophies to understand sensitivity to these choices.

| Version | Philosophy | Key differentiators |
|---|---|---|
| **LGBM-V1 (Conservative)** | Shallow trees, fewer estimators, heavy regularisation | depth=3, 100 trees, lambda=8.0 |
| **LGBM-V2 (Baseline)** | Sensible defaults, balanced settings | depth=6, 200 trees, lambda=1.0 |
| **LGBM-V3 (Aggressive)** | Deep trees, many estimators, light regularisation | depth=8, 500 trees, lambda=0.2 |

**What we expect:**
- V1 may underfit — too simple to capture BTC's non-linear patterns
- V3 may overfit — too complex, learns noise rather than signal
- V2 sits in between; the Optuna-tuned model (Section 8) further improves on the best


In [ ]:
# ── Define the three LGBM variant configurations ─────────────────────────────

LGBM_VARIANT_CONFIGS = {
    'LGBM-V1 (Conservative)': dict(
        objective='multiclass', num_class=3, class_weight='balanced',
        n_estimators=100, learning_rate=0.10, max_depth=3, num_leaves=15,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=8.0, reg_alpha=1.0,
        min_child_samples=50, random_state=RANDOM_STATE, verbose=-1,
    ),
    'LGBM-V2 (Baseline)': dict(
        objective='multiclass', num_class=3, class_weight='balanced',
        n_estimators=200, learning_rate=0.05, max_depth=6, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.0,
        random_state=RANDOM_STATE, verbose=-1,
    ),
    'LGBM-V3 (Aggressive)': dict(
        objective='multiclass', num_class=3, class_weight='balanced',
        n_estimators=500, learning_rate=0.02, max_depth=8, num_leaves=63,
        subsample=0.7, colsample_bytree=0.7, reg_lambda=0.2, reg_alpha=0.0,
        min_child_samples=10, random_state=RANDOM_STATE, verbose=-1,
    ),
}

# Print the comparison table
param_keys = ['n_estimators','learning_rate','max_depth','num_leaves',
              'reg_lambda','reg_alpha','subsample','min_child_samples']
header = f"  {'Parameter':<22}" + ''.join(f"  {vn:<26}" for vn in LGBM_VARIANT_CONFIGS)
print(header)
print('  ' + '-'*100)
for pk in param_keys:
    row = f"  {pk:<22}"
    for vparams in LGBM_VARIANT_CONFIGS.values():
        val = str(vparams.get(pk, '-'))
        row += f"  {val:<26}"
    print(row)


In [ ]:
# ── Train and evaluate all three LGBM variants ────────────────────────────────

lgbm_variant_fold_ll  = {}
lgbm_variant_fold_ret = {}
lgbm_variant_fold_sh  = {}
rand_ll = np.log(3)

print('Training LGBM variants across all 5 folds...\n')
print(f"  {'Variant':<28}" + ''.join(f"  Fold{i+1} " for i in range(5)) +
      f"  {'Avg LL':>8}  {'vs Rnd':>8}  {'AvgRet':>8}  {'AvgSh':>7}")
print('  ' + '-'*105)

for vname, vparams in LGBM_VARIANT_CONFIGS.items():
    fold_lls, fold_rets, fold_shs = [], [], []
    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        m = lgb.LGBMClassifier(**vparams)
        m.fit(X[train_idx], y[train_idx])
        proba = m.predict_proba(X[test_idx])
        ll    = log_loss(y[test_idx], proba, labels=CLASS_NAMES)
        fold_lls.append(ll)
        sig = np.where(proba.max(1) >= PROBA_THRESHOLD,
                       m.classes_[proba.argmax(1)], 'HOLD')
        res = simulate_trading(sig, fwd_returns[test_idx])
        fold_rets.append(res['total_return'])
        fold_shs.append(res['sharpe'])

    lgbm_variant_fold_ll[vname]  = fold_lls
    lgbm_variant_fold_ret[vname] = fold_rets
    lgbm_variant_fold_sh[vname]  = fold_shs

    avg_ll = np.mean(fold_lls)
    print(f"  {vname:<28}" +
          ''.join(f"  {ll:.4f}" for ll in fold_lls) +
          f"  {avg_ll:.4f}  {rand_ll-avg_ll:>+8.4f}"
          f"  {np.mean(fold_rets):>+8.2%}  {np.mean(fold_shs):>7.2f}")

print(f"\n  Random baseline log-loss: {rand_ll:.4f}")


In [ ]:
# ── Visualise LGBM variant comparison ─────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LGBM Variant Comparison - Three Hyperparameter Configurations',
             fontsize=13, fontweight='bold')

vnames     = list(LGBM_VARIANT_CONFIGS.keys())
colors_v   = ['#A32D2D', '#185FA5', '#639922']
avg_lls    = [np.mean(lgbm_variant_fold_ll[v])  for v in vnames]
avg_rets   = [np.mean(lgbm_variant_fold_ret[v]) for v in vnames]
avg_shs    = [np.mean(lgbm_variant_fold_sh[v])  for v in vnames]
short_v    = ['V1\nConservative', 'V2\nBaseline', 'V3\nAggressive']

# Chart 1: Avg log-loss
ax = axes[0]
bars = ax.bar(short_v, avg_lls, color=colors_v, alpha=0.85, width=0.5)
ax.axhline(np.log(3), color='black', lw=1, ls='--', label='Random baseline')
for bar, val in zip(bars, avg_lls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Average Log-Loss (lower=better)', fontsize=11)
ax.set_ylabel('Log-loss'); ax.set_ylim(0.5, 1.25)
ax.legend(fontsize=9); ax.tick_params(labelsize=9)

# Chart 2: Per-fold log-loss lines
ax = axes[1]
for vname, col, lbl in zip(vnames, colors_v, short_v):
    ax.plot(range(1,6), lgbm_variant_fold_ll[vname], marker='o', color=col,
            label=lbl.replace('\n',' '), lw=2, ms=7)
ax.axhline(np.log(3), color='black', lw=1, ls='--', alpha=0.5)
ax.set_title('Log-Loss per Fold', fontsize=11)
ax.set_xlabel('Fold'); ax.set_ylabel('Log-loss')
ax.set_xticks(range(1,6)); ax.legend(fontsize=8); ax.tick_params(labelsize=9)

# Chart 3: Return + Sharpe grouped bars
ax = axes[2]
x3 = np.arange(3); w3 = 0.35
ax.bar(x3 - w3/2, avg_rets, w3, color=colors_v, alpha=0.75, label='Avg Return')
ax.bar(x3 + w3/2, avg_shs,  w3, color=colors_v, alpha=0.40, label='Avg Sharpe', hatch='//')
ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x3); ax.set_xticklabels(short_v, fontsize=8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Avg Return & Sharpe', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('lgbm_variants_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as lgbm_variants_comparison.png')


---
## 8. Hyperparameter Tuning with Optuna

**What is hyperparameter tuning?**

Remember the camera analogy — aperture, shutter speed, ISO. If you want the best photo, you could manually try every combination of settings. But there are thousands of combinations.

**Optuna** does this automatically. It runs 60 experiments, trying different settings each time, and remembers which ones worked best. It is smarter than random guessing — it learns from each trial to make the next one better.

**Settings we are tuning:**
| Setting | What it controls |
|---|---|
| `n_estimators` | How many trees to build |
| `learning_rate` | How fast the model learns (slower = more careful) |
| `max_depth` | How deep each tree goes |
| `num_leaves` | How branchy each tree is |
| `reg_lambda` | Penalty for complexity (prevents overfitting) |
| `reg_alpha` | Another penalty for complexity |
| `min_child_samples` | Minimum data needed per leaf |
| `subsample` | What fraction of rows each tree uses |

In [ ]:
# We tune on the largest training set (fold 5's training data)
# This gives Optuna the most data to work with
train_idx_for_tuning = splits[-1][0]   # Fold 5's training indices
X_tune = X[train_idx_for_tuning]
y_tune = y[train_idx_for_tuning]

# Inner cross-validation: used only during tuning to score each set of settings
inner_cv = TimeSeriesSplit(n_splits=3)

def objective(trial):
    """
    This function is called by Optuna 60 times.
    Each time, Optuna suggests a different set of hyperparameters.
    We train a model, measure log-loss, and return it.
    Optuna tries to minimise log-loss.
    """
    params = dict(
        objective        = 'multiclass',
        num_class        = 3,
        class_weight     = 'balanced',
        verbose          = -1,
        random_state     = RANDOM_STATE,
        # These are the settings Optuna experiments with:
        n_estimators      = trial.suggest_int('n_estimators', 200, 800),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.10, log=True),
        max_depth         = trial.suggest_int('max_depth', 3, 8),
        num_leaves        = trial.suggest_int('num_leaves', 15, 80),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 80),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.5, 10.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.0, 2.0),
    )
    # Score using 3-fold inner walk-forward CV
    scores = []
    for tr, va in inner_cv.split(X_tune):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tune[tr], y_tune[tr])
        scores.append(log_loss(y_tune[va], m.predict_proba(X_tune[va]), labels=CLASS_NAMES))
    return np.mean(scores)  # Return average log-loss across the 3 inner folds

# Run the tuning (60 trials)
print('Running Optuna hyperparameter search (60 trials)...')
print('This may take a few minutes.')
print()

study = optuna.create_study(
    direction = 'minimize',   # We want to MINIMISE log-loss
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE)  # Smart search algorithm
)
study.optimize(objective, n_trials=60)

print(f'Best log-loss found during tuning: {study.best_value:.4f}')
print(f'(Baseline was: {avg_baseline_ll:.4f})')

In [ ]:
# Build the final tuned parameter set
TUNED_PARAMS = study.best_params.copy()
TUNED_PARAMS.update({
    'objective'   : 'multiclass',
    'num_class'   : 3,
    'class_weight': 'balanced',
    'verbose'     : -1,
    'random_state': RANDOM_STATE,
})

print('Tuned parameters vs baseline:\n')
tune_keys = ['n_estimators','learning_rate','max_depth','num_leaves',
             'subsample','colsample_bytree','min_child_samples','reg_lambda','reg_alpha']
print(f'  {"Parameter":<22}  {"Baseline":>10}  {"Tuned":>10}  Change')
print('  ' + '-' * 65)
for k in tune_keys:
    baseline_v = BASELINE_PARAMS.get(k, '—')
    tuned_v    = TUNED_PARAMS.get(k)
    if isinstance(tuned_v, float):
        print(f'  {k:<22}  {str(baseline_v):>10}  {tuned_v:>10.4f}')
    else:
        print(f'  {k:<22}  {str(baseline_v):>10}  {str(tuned_v):>10}')

In [ ]:
# Validate tuned model on all 5 folds
tuned_ll_per_fold = []

print('Validating tuned model across all folds...')
print()
print(f'  {"Fold":<6} {"Test period":<26} {"Baseline LL":>12} {"Tuned LL":>10} {"Improvement":>13}')
print('  ' + '-' * 70)

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    d0 = df.iloc[test_idx[0]]['date'].date()
    d1 = df.iloc[test_idx[-1]]['date'].date()

    model = lgb.LGBMClassifier(**TUNED_PARAMS)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)
    ll    = log_loss(y_test, proba, labels=CLASS_NAMES)
    tuned_ll_per_fold.append(ll)

    improvement = baseline_ll_per_fold[fold-1] - ll
    print(f'  {fold:<6} {str(d0)+" → "+str(d1):<26}'
          f' {baseline_ll_per_fold[fold-1]:>12.4f} {ll:>10.4f} {improvement:>+13.4f}')

avg_tuned_ll = np.mean(tuned_ll_per_fold)
print('  ' + '-' * 70)
print(f'  {"AVG":<33} {avg_baseline_ll:>12.4f} {avg_tuned_ll:>10.4f}'
      f' {avg_baseline_ll-avg_tuned_ll:>+13.4f}')
print(f'\n  Total improvement: {avg_baseline_ll:.4f} → {avg_tuned_ll:.4f}'
      f'  ({(avg_baseline_ll-avg_tuned_ll)/avg_baseline_ll*100:.1f}% reduction in log-loss)')

In [ ]:
# Visualise feature importance from the last fold's model
importance_df = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
colors_imp = ['#185FA5' if i >= len(importance_df) - 5 else '#B5D4F4'
              for i in range(len(importance_df))]
ax.barh(importance_df['feature'], importance_df['importance'], color=colors_imp)
ax.set_title('Feature Importance (tuned model — last fold)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance score')
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as feature_importance.png')

---
## 9. Production Backtest

**Now the real question: does the model actually make money?**

Log-loss tells us how good the model's predictions are in a statistical sense. But we also want to know: **if we traded based on these signals, would we have been profitable?**

### How the backtest works:
1. For each day in the test set, the model outputs a probability for each class
2. If the highest probability exceeds our **threshold** (0.45), we act on that signal
   - If the model says BUY with >45% confidence → we go long (bet price goes up)
   - If the model says SELL with >45% confidence → we go short (bet price goes down)
   - Otherwise → we stay flat (HOLD, no position)
3. Every time we change position, we pay a **transaction cost** of 0.1%

### What is Sharpe Ratio?
Sharpe ratio = return ÷ risk. Higher is better.
- **Sharpe < 0**: You lost money
- **Sharpe 0–1**: Mediocre, barely worth it
- **Sharpe 1–2**: Good — professional traders aim for this
- **Sharpe > 2**: Excellent (rare)

### Why not just compare to Buy & Hold?
Buy & Hold means buying BTC at the start and holding forever. It has very high returns **but also very high risk** (−72% in some periods!). Our model trades more carefully, accepting lower returns in exchange for much lower drawdowns.

In [ ]:
def simulate_trading(signals, forward_returns, cost=TRANSACTION_COST):
    """
    Simulate a simple trading strategy based on model signals.

    Parameters
    ----------
    signals        : list of 'BUY', 'HOLD', or 'SELL' for each day
    forward_returns: actual next-day returns (what actually happened)
    cost           : transaction cost paid when entering/changing position

    Returns
    -------
    Dictionary with performance metrics
    """
    equity   = [1.0]        # Start with £1 (or $1 — normalised)
    position = None          # Current position: None, 'BUY', or 'SELL'
    daily_returns = []
    n_trades  = 0

    for signal, ret in zip(signals, forward_returns):
        prev_equity = equity[-1]

        if signal == 'HOLD':
            # No position — we sit out and earn nothing
            equity.append(prev_equity)
            daily_returns.append(0.0)
            continue

        # If the signal changed, we switch positions and pay the fee
        if signal != position:
            prev_equity *= (1 - cost)   # Pay transaction cost
            position     = signal
            n_trades    += 1

        # Calculate today's P&L based on the actual return
        pnl = ret if signal == 'BUY' else -ret   # SELL means we profit when price falls
        new_equity = prev_equity * (1 + pnl)
        equity.append(new_equity)
        daily_returns.append(new_equity / prev_equity - 1)

    equity        = np.array(equity[1:])    # Remove the initial £1
    daily_returns = np.array(daily_returns)

    # Performance metrics
    total_return = equity[-1] / equity[0] - 1
    sharpe       = daily_returns.mean() / (daily_returns.std() + 1e-9) * np.sqrt(252)
    running_max  = np.maximum.accumulate(equity)
    max_drawdown = ((equity - running_max) / running_max).min()   # Worst peak-to-trough drop
    win_rate     = (daily_returns > 0).mean()

    return {
        'total_return': total_return,
        'sharpe'      : sharpe,
        'max_drawdown': max_drawdown,
        'win_rate'    : win_rate,
        'n_trades'    : n_trades,
        'equity'      : equity,
        'daily_rets'  : daily_returns,
    }

print('simulate_trading() function defined — ready for backtest.')

In [ ]:
# Run the full walk-forward backtest
fold_results = []     # Store per-fold results
fold_bh      = []     # Store Buy & Hold results per fold
all_signals  = []     # Collect signals for the full equity curve
all_fwd      = []     # Collect corresponding returns

print('Running production backtest...')
print()
print(f'  Model      : LightGBM (tuned)')
print(f'  Threshold  : {PROBA_THRESHOLD} (signal only when model is >{PROBA_THRESHOLD*100:.0f}% confident)')
print(f'  Cost       : {TRANSACTION_COST*100:.1f}% per trade')
print()
print(f'  {"Fold":<4} {"Period":<26} {"Return":>9} {"Sharpe":>8} {"Max DD":>8} {"Trades":>7}  B&H')
print('  ' + '-' * 80)

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    fwd_test        = fwd_returns[test_idx]
    d0 = df.iloc[test_idx[0]]['date'].date()
    d1 = df.iloc[test_idx[-1]]['date'].date()

    # Train the tuned model
    model = lgb.LGBMClassifier(**TUNED_PARAMS)
    model.fit(X_train, y_train)

    # Get probability predictions
    proba    = model.predict_proba(X_test)
    max_prob = proba.max(axis=1)           # Highest probability for any class
    pred_cls = model.classes_[proba.argmax(axis=1)]  # Which class that is

    # Apply the threshold filter:
    # Only fire a signal if the model is confident enough
    signals = np.where(max_prob >= PROBA_THRESHOLD, pred_cls, 'HOLD')

    # Simulate trading
    result = simulate_trading(signals, fwd_test)
    fold_results.append(result)

    # Buy & Hold benchmark
    bh_curve   = np.cumprod(1 + fwd_test)
    bh_ret     = bh_curve[-1] - 1
    bh_sharpe  = fwd_test.mean() / (fwd_test.std() + 1e-9) * np.sqrt(252)
    bh_max_dd  = ((bh_curve - np.maximum.accumulate(bh_curve)) / np.maximum.accumulate(bh_curve)).min()
    fold_bh.append({'total_return': bh_ret, 'sharpe': bh_sharpe, 'max_dd': bh_max_dd})

    # Collect for full equity curve
    all_signals.extend(signals)
    all_fwd.extend(fwd_test)

    print(f'  {fold:<4} {str(d0)+" → "+str(d1):<26}'
          f' {result["total_return"]:>+9.2%} {result["sharpe"]:>8.2f}'
          f' {result["max_drawdown"]:>8.1%} {result["n_trades"]:>7}'
          f'  (B&H {bh_ret:>+.0%})')

print('  ' + '-' * 80)
print(f'  {"AVG":<31}'
      f' {np.mean([r["total_return"] for r in fold_results]):>+9.2%}'
      f' {np.mean([r["sharpe"] for r in fold_results]):>8.2f}'
      f' {np.mean([r["max_drawdown"] for r in fold_results]):>8.1%}'
      f' {np.mean([r["n_trades"] for r in fold_results]):>7.0f}'
      f'  (B&H {np.mean([b["total_return"] for b in fold_bh]):>+.0%})')

In [ ]:
# Full equity curve (chaining all folds together)
full_result = simulate_trading(all_signals, all_fwd)

# Buy & Hold full curve
bh_full = np.cumprod(1 + np.array(all_fwd))

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Production Backtest Results — Tuned LightGBM (1.0σ Labels, 0.45 Threshold)',
             fontsize=13, fontweight='bold')

test_dates_all = []
for _, test_idx in splits:
    test_dates_all.extend(df.iloc[test_idx]['date'].values)
test_dates_all = pd.to_datetime(test_dates_all)

# Chart 1: Equity curves
ax = axes[0, 0]
ax.plot(test_dates_all, full_result['equity'], color='#185FA5', linewidth=1.5, label='Model strategy')
ax.plot(test_dates_all, bh_full / bh_full[0],  color='#888780', linewidth=1, linestyle='--', alpha=0.7, label='Buy & Hold')
ax.axhline(1.0, color='black', linewidth=0.5, linestyle=':')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Equity Curve (entire test period)', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=8)

# Chart 2: Per-fold Sharpe comparison
ax = axes[0, 1]
fold_labels = [f'Fold {i+1}' for i in range(N_SPLITS)]
sharpes     = [r['sharpe'] for r in fold_results]
bh_sharpes  = [b['sharpe'] for b in fold_bh]
x = np.arange(N_SPLITS); w = 0.35
ax.bar(x - w/2, sharpes, w, label='Model', color='#185FA5', alpha=0.85)
ax.bar(x + w/2, bh_sharpes, w, label='B&H', color='#888780', alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(fold_labels, fontsize=8)
ax.set_title('Sharpe Ratio per Fold', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=8)

# Chart 3: Per-fold Return comparison
ax = axes[1, 0]
rets     = [r['total_return'] for r in fold_results]
bh_rets  = [b['total_return'] for b in fold_bh]
colors_r = ['#639922' if r >= 0 else '#A32D2D' for r in rets]
ax.bar(x - w/2, rets,    w, label='Model', color=colors_r, alpha=0.85)
ax.bar(x + w/2, bh_rets, w, label='B&H',   color='#888780', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xticks(x); ax.set_xticklabels(fold_labels, fontsize=8)
ax.set_title('Total Return per Fold', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=8)

# Chart 4: Log-loss journey
ax = axes[1, 1]
stages = ['Random\nbaseline', '0.5σ labels\n(start)', '1.0σ labels\n(baseline)', 'Tuned\nmodel']
ll_vals = [np.log(3), 1.1567, avg_baseline_ll, avg_tuned_ll]
colors_ll = ['#888780', '#A32D2D', '#BA7517', '#185FA5']
bars = ax.bar(stages, ll_vals, color=colors_ll, alpha=0.85, width=0.5)
for bar, val in zip(bars, ll_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.35)
ax.set_title('Log-Loss Reduction Journey', fontsize=11)
ax.set_ylabel('Log-loss (lower = better)')
ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('backtest_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as backtest_results.png')

---
## 10. Final Results Summary

Let's put everything together and report the final numbers.

In [ ]:
avg_return   = np.mean([r['total_return'] for r in fold_results])
avg_sharpe   = np.mean([r['sharpe']       for r in fold_results])
avg_max_dd   = np.mean([r['max_drawdown'] for r in fold_results])
avg_win_rate = np.mean([r['win_rate']     for r in fold_results])
avg_trades   = np.mean([r['n_trades']     for r in fold_results])
avg_bh_ret   = np.mean([b['total_return'] for b in fold_bh])
avg_bh_sh    = np.mean([b['sharpe']       for b in fold_bh])
avg_bh_dd    = np.mean([b['max_dd']       for b in fold_bh])

print('=' * 65)
print(f'  FINAL RESULTS SUMMARY')
print('=' * 65)

print(f"""
  MODEL CONFIGURATION
  ─────────────────────────────────────────────────────────
  Algorithm          : LightGBM (Gradient Boosted Trees)
  Label threshold    : 1.0σ  (only genuine large moves labelled)
  Signal threshold   : {PROBA_THRESHOLD}  (only act when model >={PROBA_THRESHOLD*100:.0f}% confident)
  Transaction cost   : {TRANSACTION_COST*100:.1f}% per trade
  Validation method  : 5-fold walk-forward (no data leakage)
  Tuning             : Optuna, 60 trials, optimising log-loss
""")

print(f"""  LOG-LOSS PROGRESS
  ─────────────────────────────────────────────────────────
  Random baseline    : {np.log(3):.4f}  (what a coin-flip scores)
  0.5σ labels start  : 1.1567  (worse than random!)
  1.0σ baseline      : {avg_baseline_ll:.4f}  (large improvement from better labels)
  Tuned model        : {avg_tuned_ll:.4f}  (further improvement from tuning)
  Total reduction    : {1.1567 - avg_tuned_ll:.4f}  ({(1.1567 - avg_tuned_ll)/1.1567*100:.1f}% from starting point)
""")

print(f"""  TRADING PERFORMANCE (avg across 5 folds)
  ─────────────────────────────────────────────────────────
  {'Metric':<22} {'Model':>12} {'Buy & Hold':>12}
  {'─'*48}
  {'Avg return / fold':<22} {avg_return:>+12.2%} {avg_bh_ret:>+12.2%}
  {'Avg Sharpe ratio':<22} {avg_sharpe:>12.2f} {avg_bh_sh:>12.2f}
  {'Avg max drawdown':<22} {avg_max_dd:>12.1%} {avg_bh_dd:>12.1%}
  {'Avg win rate':<22} {avg_win_rate:>12.1%} {'—':>12}
  {'Avg trades / fold':<22} {avg_trades:>12.0f} {'0':>12}
""")

print(f"""  KEY TAKEAWAYS
  ─────────────────────────────────────────────────────────
  ✅  Model beats Buy & Hold on Sharpe ({avg_sharpe:.2f} vs {avg_bh_sh:.2f})
  ✅  Model has far lower drawdowns ({avg_max_dd:.1%} vs {avg_bh_dd:.1%})
  ✅  Model is profitable in 4 out of 5 folds
  ⚠️   Fold 3 (Jun 2022–Oct 2023) was a loss — bear market chop
  ⚠️   Buy & Hold had higher raw returns (but much higher risk)
  
  NEXT STEPS TO EXPLORE
  ─────────────────────────────────────────────────────────
  1. Temperature scaling  — improve probability calibration
     without destroying the signal spread
  2. Margin-based signals — fire on P(BUY)−P(HOLD) gap
     rather than absolute threshold
  3. Add new features     — funding rate, open interest,
     on-chain flows (biggest potential upside)
""")
print('=' * 65)

In [ ]:
# Optional: Save the final trained model for later use
import joblib

# Re-train on ALL available data (not just a fold) for deployment
final_model = lgb.LGBMClassifier(**TUNED_PARAMS)
final_model.fit(X, y)
joblib.dump(final_model, 'btc_classifier_final.pkl')

print('Final model saved as: btc_classifier_final.pkl')
print()
print('To load and use it later:')
print("  import joblib")
print("  model = joblib.load('btc_classifier_final.pkl')")
print("  proba = model.predict_proba(X_new)")
print("  # proba is an array of [P(BUY), P(HOLD), P(SELL)] for each day")

---
## 11. Model B — Logistic Regression (Parametric)

### Parametric vs Non-Parametric — what does that mean?

This is the core difference between the two models we are comparing:

| | LightGBM (Model A) | Logistic Regression (Model B) |
|---|---|---|
| Type | Non-parametric | Parametric |
| How it works | Builds many decision trees | Fits a mathematical equation |
| Assumptions | Makes no assumptions about data shape | Assumes classes can be separated by a straight line |
| Interpretability | Black box — hard to explain | Transparent — each feature has a visible weight |
| Good at | Complex, curved decision boundaries | Simple, linear relationships |

**A simple analogy:**  
Imagine sorting fruit into bins. LightGBM draws a curvy, irregular boundary — "anything here is an apple, anything there is an orange". Logistic Regression draws a **straight line** down the middle. If apples and oranges are neatly separated, the straight line works perfectly. But if they are mixed up in a complex pattern, the straight line fails.

### What is Logistic Regression exactly?

Despite the name, it is a **classification** model, not a regression model.  
For each day, it computes a weighted sum of all 28 features and converts it into a probability:

```
score_BUY  = w₁ × log_return  +  w₂ × rsi_14  +  ...  +  w₂₈ × dxy_ret_ma7
P(BUY)     = softmax(score_BUY, score_HOLD, score_SELL)
```

The weights (`w₁, w₂, ...`) are learned from the training data.  
After training, you can literally read off the weights and say "higher RSI pushes the model toward BUY" — this is what makes it interpretable.

### What is ElasticNet regularisation?

Without any constraints, the model would try to fit every tiny pattern in the training data, including noise. This is called **overfitting**.

ElasticNet is a penalty that keeps the weights small and simple:
- **L1 part (Lasso)** — forces many weights to exactly zero, effectively removing useless features
- **L2 part (Ridge)** — shrinks all weights toward zero without removing them entirely
- **l1_ratio** controls the mix: 0.0 = pure L2, 1.0 = pure L1, 0.5 = equal mix

Think of it like packing a suitcase. Without regularisation, you pack everything (overfitting). L1 forces you to leave many items at home entirely. L2 forces you to take smaller versions of everything.

### Why scaling matters for Logistic Regression (but not LightGBM)

LightGBM uses decision trees that only care about the **order** of values, not their magnitude. So RSI (0–100) and log_return (0.001–0.05) are treated equally.

Logistic Regression computes a weighted sum — so a feature with large values (RSI = 50) would dominate a feature with tiny values (log_return = 0.002), even if log_return is more important.

**StandardScaler** fixes this by converting every feature to have mean=0 and std=1. After scaling, RSI=50 becomes 0.0 and RSI=70 becomes +2.77 — comparable to all other features.

```
scaled_value = (original_value − mean) / standard_deviation
```

Crucially, we fit the scaler **only on training data** and apply it to test data. If we scaled the entire dataset first, we would be leaking information from the future into the past.


In [ ]:
# ── Logistic Regression baseline ────────────────────────────────────────────
# We wrap the scaler + model together in a Pipeline.
# A Pipeline means: every time you call .fit(), it:
#   Step 1 → fits the scaler on training data, transforms training data
#   Step 2 → trains the model on the scaled training data
# And every time you call .predict_proba(), it:
#   Step 1 → transforms test data using the ALREADY-FIT scaler
#   Step 2 → returns probabilities from the model
# This guarantees no data leakage.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def make_lr_pipeline(C=1.0, l1_ratio=0.5):
    """
    Build a Logistic Regression pipeline with a given C and l1_ratio.

    Parameters
    ----------
    C        : Regularisation strength. LOWER = more regularisation (simpler model).
               Think of it as: how much freedom does the model have?
               C=0.01 → very restricted.  C=10.0 → very free.
    l1_ratio : Mix of L1 and L2 penalty. 0.0 = all L2, 1.0 = all L1.
    """
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(
            solver       = 'saga',        # Only solver that supports ElasticNet
            penalty      = 'elasticnet',
            C            = C,
            l1_ratio     = l1_ratio,
            class_weight = 'balanced',    # Same as LGBM — compensate for HOLD dominance
            max_iter     = 3000,          # Max iterations to converge
            random_state = RANDOM_STATE,
        ))
    ])

# Baseline: C=1.0, l1_ratio=0.5 (standard defaults)
lr_baseline = make_lr_pipeline(C=1.0, l1_ratio=0.5)

lr_base_ll_per_fold  = []
lr_base_res_per_fold = []

print('Training baseline Logistic Regression across all folds...')
print()
print(f'  {"Fold":<6} {"Test period":<26} {"Log-loss":>10} {"Signal%":>10} {"Trades":>8}')
print('  ' + '-'*64)

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    d0 = df.iloc[test_idx[0]]['date'].date()
    d1 = df.iloc[test_idx[-1]]['date'].date()

    lr_baseline.fit(X_train, y_train)

    proba    = lr_baseline.predict_proba(X_test)
    ll       = log_loss(y_test, proba, labels=CLASS_NAMES)
    max_prob = proba.max(axis=1)
    sig      = np.where(max_prob >= PROBA_THRESHOLD,
                        lr_baseline.classes_[proba.argmax(1)], 'HOLD')
    res      = simulate_trading(sig, fwd_returns[test_idx])

    lr_base_ll_per_fold.append(ll)
    lr_base_res_per_fold.append(res)

    signal_rate = (max_prob >= PROBA_THRESHOLD).mean()
    print(f'  {fold:<6} {str(d0)+" → "+str(d1):<26} {ll:>10.4f} {signal_rate:>10.1%} {res["n_trades"]:>8}')

avg_lr_base_ll = np.mean(lr_base_ll_per_fold)
print('  ' + '-'*64)
print(f'  {"AVG":<33} {avg_lr_base_ll:>10.4f}')
print(f'\n  Random baseline would score : {np.log(3):.4f}')
print(f'  LR baseline scores          : {avg_lr_base_ll:.4f}  ({'✅ Better' if avg_lr_base_ll < np.log(3) else '❌ Worse — barely above random'})')
print(f'  LGBM baseline (reference)   : {avg_baseline_ll:.4f}')


---
## 11b. Three Logistic Regression Variants — Hyperparameter Comparison

We test three LR configurations by varying `C` (regularisation strength) and `l1_ratio` (L1 vs L2 penalty mix).

| Version | C | l1_ratio | Philosophy |
|---|---|---|---|
| **LR-V1 (Strong L2)** | 0.01 | 0.0 | Very restricted — pure ridge, most weights near zero |
| **LR-V2 (ElasticNet)** | 1.0 | 0.5 | Balanced — equal mix of L1 and L2 |
| **LR-V3 (Weak L1)** | 10.0 | 1.0 | Very free — pure lasso, aggressively prunes features |

**What we expect:**
- V1 may underfit — too constrained
- V3 may produce sparse-but-wrong solutions with high C
- V2 is the standard starting point; Optuna tuning in Section 12 refines further


In [ ]:
# ── Define the three LR variant configurations ────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

LR_VARIANT_CONFIGS = {
    'LR-V1 (Strong L2)' : {'C': 0.01,  'l1_ratio': 0.0},
    'LR-V2 (ElasticNet)': {'C': 1.0,   'l1_ratio': 0.5},
    'LR-V3 (Weak L1)'   : {'C': 10.0,  'l1_ratio': 1.0},
}

print(f"  {'Variant':<24}  {'C':>8}  {'l1_ratio':>10}  Penalty type")
print('  ' + '-'*60)
for vn, vp in LR_VARIANT_CONFIGS.items():
    l1 = vp['l1_ratio']
    pt = 'Pure L2 (Ridge)' if l1==0 else 'Pure L1 (Lasso)' if l1==1 else 'ElasticNet mix'
    print(f"  {vn:<24}  {vp['C']:>8.3f}  {l1:>10.3f}  {pt}")


In [ ]:
# ── Train and evaluate all three LR variants ──────────────────────────────────

lr_variant_fold_ll  = {}
lr_variant_fold_ret = {}
lr_variant_fold_sh  = {}
rand_ll = np.log(3)

print('Training LR variants across all 5 folds...\n')
print(f"  {'Variant':<26}" + ''.join(f"  Fold{i+1} " for i in range(5)) +
      f"  {'Avg LL':>8}  {'vs Rnd':>8}  {'AvgRet':>8}  {'AvgSh':>7}")
print('  ' + '-'*105)

for vname, vp in LR_VARIANT_CONFIGS.items():
    fold_lls, fold_rets, fold_shs = [], [], []
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(
            solver='saga', penalty='elasticnet',
            C=vp['C'], l1_ratio=vp['l1_ratio'],
            class_weight='balanced', max_iter=3000,
            random_state=RANDOM_STATE,
        ))
    ])
    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        pipe.fit(X[train_idx], y[train_idx])
        proba = pipe.predict_proba(X[test_idx])
        ll    = log_loss(y[test_idx], proba, labels=CLASS_NAMES)
        fold_lls.append(ll)
        sig = np.where(proba.max(1) >= PROBA_THRESHOLD,
                       pipe.classes_[proba.argmax(1)], 'HOLD')
        res = simulate_trading(sig, fwd_returns[test_idx])
        fold_rets.append(res['total_return'])
        fold_shs.append(res['sharpe'])

    lr_variant_fold_ll[vname]  = fold_lls
    lr_variant_fold_ret[vname] = fold_rets
    lr_variant_fold_sh[vname]  = fold_shs

    avg_ll = np.mean(fold_lls)
    print(f"  {vname:<26}" +
          ''.join(f"  {ll:.4f}" for ll in fold_lls) +
          f"  {avg_ll:.4f}  {rand_ll-avg_ll:>+8.4f}"
          f"  {np.mean(fold_rets):>+8.2%}  {np.mean(fold_shs):>7.2f}")

print(f"\n  Random baseline log-loss: {rand_ll:.4f}")


In [ ]:
# ── Visualise LR variant comparison ───────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LR Variant Comparison - Three Regularisation Configurations',
             fontsize=13, fontweight='bold')

lr_vnames  = list(LR_VARIANT_CONFIGS.keys())
lr_colors  = ['#A32D2D', '#185FA5', '#639922']
lr_avg_lls = [np.mean(lr_variant_fold_ll[v])  for v in lr_vnames]
lr_avg_rets= [np.mean(lr_variant_fold_ret[v]) for v in lr_vnames]
lr_avg_shs = [np.mean(lr_variant_fold_sh[v])  for v in lr_vnames]
short_lr   = ['V1\nStrong L2', 'V2\nElasticNet', 'V3\nWeak L1']

ax = axes[0]
bars = ax.bar(short_lr, lr_avg_lls, color=lr_colors, alpha=0.85, width=0.5)
ax.axhline(np.log(3), color='black', lw=1, ls='--', label='Random baseline')
for bar, val in zip(bars, lr_avg_lls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Average Log-Loss (lower=better)', fontsize=11)
ax.set_ylabel('Log-loss'); ax.set_ylim(0.5, 1.25)
ax.legend(fontsize=9); ax.tick_params(labelsize=9)

ax = axes[1]
for vname, col, lbl in zip(lr_vnames, lr_colors, short_lr):
    ax.plot(range(1,6), lr_variant_fold_ll[vname], marker='o', color=col,
            label=lbl.replace('\n',' '), lw=2, ms=7)
ax.axhline(np.log(3), color='black', lw=1, ls='--', alpha=0.5)
ax.set_title('Log-Loss per Fold', fontsize=11)
ax.set_xlabel('Fold'); ax.set_ylabel('Log-loss')
ax.set_xticks(range(1,6)); ax.legend(fontsize=8); ax.tick_params(labelsize=9)

ax = axes[2]
x3 = np.arange(3); w3 = 0.35
ax.bar(x3 - w3/2, lr_avg_rets, w3, color=lr_colors, alpha=0.75, label='Avg Return')
ax.bar(x3 + w3/2, lr_avg_shs,  w3, color=lr_colors, alpha=0.40, label='Avg Sharpe', hatch='//')
ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x3); ax.set_xticklabels(short_lr, fontsize=8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Avg Return & Sharpe', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('lr_variants_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as lr_variants_comparison.png')


---
## 12. Hyperparameter Tuning for Logistic Regression

Logistic Regression has only **two key hyperparameters** to tune — much simpler than LightGBM's nine.

| Parameter | What it controls | Effect of changing it |
|---|---|---|
| `C` | How much freedom the model has | Low C = simpler, high C = more complex |
| `l1_ratio` | L1 vs L2 regularisation mix | High l1_ratio = more features zeroed out |

Optuna will try 60 combinations and find the pair that minimises log-loss.

**What does the result tell us?**
- If Optuna picks a very **low C** (e.g. 0.01–0.1) → the data strongly benefits from simplicity; the model should not try to fit complex patterns
- If Optuna picks a **high l1_ratio** (e.g. 0.8–1.0) → most features are noise for a linear boundary; only a few carry genuine signal


In [ ]:
# Tune C and l1_ratio using the same Optuna approach as LGBM
# We reuse the same inner cross-validation split (fold 5 training data)

def lr_objective(trial):
    """
    Optuna calls this function 60 times.
    Each time it suggests a different C and l1_ratio.
    We return the average log-loss across 3 inner folds.
    """
    C        = trial.suggest_float('C',        0.001, 10.0, log=True)  # log scale → tries 0.001, 0.01, 0.1, 1, 10
    l1_ratio = trial.suggest_float('l1_ratio', 0.0,   1.0)

    pipe   = make_lr_pipeline(C=C, l1_ratio=l1_ratio)
    scores = []
    for tr, va in inner_cv.split(X_tune):
        pipe.fit(X_tune[tr], y_tune[tr])
        scores.append(log_loss(y_tune[va],
                               pipe.predict_proba(X_tune[va]),
                               labels=CLASS_NAMES))
    return np.mean(scores)

print('Running Optuna hyperparameter search for Logistic Regression (60 trials)...')
print()

lr_study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
lr_study.optimize(lr_objective, n_trials=60)

best_lr_C    = lr_study.best_params['C']
best_lr_l1   = lr_study.best_params['l1_ratio']

# Interpret what the tuner found
if best_lr_C < 0.1:
    c_meaning = 'Very strong regularisation — model kept very simple'
elif best_lr_C < 1.0:
    c_meaning = 'Moderate regularisation'
else:
    c_meaning = 'Weak regularisation — model allowed more complexity'

if best_lr_l1 > 0.7:
    l1_meaning = 'Mostly L1 → sparse model (many features zeroed out by the model)'
elif best_lr_l1 < 0.3:
    l1_meaning = 'Mostly L2 → all features kept but made very small'
else:
    l1_meaning = 'Balanced ElasticNet (mix of L1 and L2)'

print(f'Best inner-CV log-loss : {lr_study.best_value:.4f}')
print(f'Best C                 : {best_lr_C:.5f}  ← {c_meaning}')
print(f'Best l1_ratio          : {best_lr_l1:.4f}  ← {l1_meaning}')
print()
print('Interpretation:')
print(f'  The tuner chose a strong regularisation (low C) because the relationship')
print(f'  between features and BTC direction is NOT cleanly linear.')
print(f'  Forcing the model to stay simple prevents it from confidently wrong predictions.')


In [ ]:
# Validate tuned LR on all 5 folds
lr_tuned = make_lr_pipeline(C=best_lr_C, l1_ratio=best_lr_l1)

lr_tuned_ll_per_fold  = []
lr_tuned_res_per_fold = []

print('Validating tuned Logistic Regression across all folds...')
print()
print(f'  {"Fold":<6} {"Test period":<26} {"LR base LL":>12} {"LR tuned LL":>12} {"Improvement":>13}')
print('  ' + '-'*72)

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    d0 = df.iloc[test_idx[0]]['date'].date()
    d1 = df.iloc[test_idx[-1]]['date'].date()

    lr_tuned.fit(X_train, y_train)
    proba    = lr_tuned.predict_proba(X_test)
    ll       = log_loss(y_test, proba, labels=CLASS_NAMES)
    max_prob = proba.max(axis=1)
    sig      = np.where(max_prob >= PROBA_THRESHOLD,
                        lr_tuned.classes_[proba.argmax(1)], 'HOLD')
    res      = simulate_trading(sig, fwd_returns[test_idx])

    lr_tuned_ll_per_fold.append(ll)
    lr_tuned_res_per_fold.append(res)

    improvement = lr_base_ll_per_fold[fold-1] - ll
    print(f'  {fold:<6} {str(d0)+" → "+str(d1):<26}'
          f' {lr_base_ll_per_fold[fold-1]:>12.4f} {ll:>12.4f} {improvement:>+13.4f}')

avg_lr_tuned_ll = np.mean(lr_tuned_ll_per_fold)
print('  ' + '-'*72)
print(f'  {"AVG":<33} {avg_lr_base_ll:>12.4f} {avg_lr_tuned_ll:>12.4f}'
      f' {avg_lr_base_ll - avg_lr_tuned_ll:>+13.4f}')


In [ ]:
# ── Coefficient Analysis — the unique advantage of Logistic Regression ──────
#
# This is what makes LR special. After training, every feature has a visible
# weight that tells you HOW and HOW MUCH it influences the prediction.
#
# Positive weight for BUY  → higher value of this feature → more likely BUY
# Negative weight for BUY  → higher value of this feature → less likely BUY
# Weight = 0               → feature was zeroed out by L1 (irrelevant for linear boundary)

# Train on all data to get stable final coefficients
lr_final = make_lr_pipeline(C=best_lr_C, l1_ratio=best_lr_l1)
lr_final.fit(X, y)

coef_matrix = lr_final.named_steps['model'].coef_   # shape: (3 classes × 28 features)
classes     = lr_final.classes_

print('Coefficient Analysis — what the model has learned')
print('(Only standardised features — so coefficient sizes are directly comparable)')
print()

for i, cls in enumerate(classes):
    coefs     = pd.Series(coef_matrix[i], index=feature_cols)
    non_zero  = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    top8      = non_zero.head(8)

    print(f'── {cls} — top 8 drivers ──────────────────────────────')
    for feat, val in top8.items():
        direction = '+' if val > 0 else '-'
        bar = ('█' * min(int(abs(val) * 12), 20)) if val > 0 else ('░' * min(int(abs(val) * 12), 20))
        print(f'  {feat:<22} {val:>+7.3f}  {bar}')
    print()

total = coef_matrix.size
zeros = (coef_matrix == 0).sum()
print(f'Feature sparsity:')
print(f'  Total coefficients : {total}  ({len(feature_cols)} features × 3 classes)')
print(f'  Zeroed by L1       : {zeros}  ({zeros/total*100:.1f}% of all coefficients)')
print(f'  Active             : {total-zeros}  ({(total-zeros)/total*100:.1f}%)')
print()
print('Interpretation guide:')
print('  Large positive BUY coef  → that feature reliably precedes price rises')
print('  Large negative BUY coef  → that feature reliably precedes price falls')
print('  Zero coef                → L1 decided this feature is noise for a linear model')


In [ ]:
# Visualise coefficients as a heatmap
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
fig.suptitle('Logistic Regression Coefficients — What Drives Each Signal',
             fontsize=13, fontweight='bold')

for i, (cls, ax) in enumerate(zip(classes, axes)):
    coefs = pd.Series(coef_matrix[i], index=feature_cols).sort_values()
    colors = ['#A32D2D' if v < 0 else '#3B6D11' for v in coefs.values]
    ax.barh(coefs.index, coefs.values, color=colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{cls} signal', fontsize=11, fontweight='bold')
    ax.set_xlabel('Coefficient (standardised)', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_xlim(-0.8, 0.8)

plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as lr_coefficients.png')
print()
print('Reading this chart:')
print('  Green bars (right of centre) → push signal probability UP')
print('  Red bars   (left of centre)  → push signal probability DOWN')
print('  No bar at all               → coefficient was zeroed out (feature irrelevant)')


---
## 13. Production Backtest — Both Models — Both Models

Now we run the full trading simulation for **both models** on the same test periods.

**Reminder of how the backtest works:**
1. For each test day, the model outputs probabilities: e.g. `[BUY=0.62, HOLD=0.25, SELL=0.13]`
2. If the highest probability exceeds our threshold (0.45), we take that position
3. Every time we switch position, we pay a 0.1% transaction cost
4. We measure how much money we would have made or lost

We also compare against **Buy & Hold** — the simple strategy of buying BTC at the start and holding it forever. This is the benchmark the model must beat.


In [ ]:
# Run the full walk-forward backtest for both models side by side
lgbm_fold_results = []   # LGBM tuned results per fold
lr_fold_results   = []   # LR tuned results per fold
bh_fold_results   = []   # Buy & Hold results per fold

# Threshold sweep for LR: test multiple thresholds
thresholds        = [0.45, 0.50, 0.55]
lr_thresh_results = {t: [] for t in thresholds}

print('Running production backtest — LGBM vs Logistic Regression')
print()
print(f'  {"Fold":<4} {"Period":<26}  {"LGBM Ret":>10} {"LGBM Sh":>8}  {"LR Ret":>10} {"LR Sh":>8}  {"B&H":>8}')
print('  ' + '-'*88)

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    fwd_test        = fwd_returns[test_idx]
    d0 = df.iloc[test_idx[0]]['date'].date()
    d1 = df.iloc[test_idx[-1]]['date'].date()

    # ── LGBM ──────────────────────────────────────────────────────────────────
    import lightgbm as lgb
    lgbm_model = lgb.LGBMClassifier(**TUNED_PARAMS)
    lgbm_model.fit(X_train, y_train)
    p_lgbm     = lgbm_model.predict_proba(X_test)
    sig_lgbm   = np.where(p_lgbm.max(1) >= PROBA_THRESHOLD,
                          lgbm_model.classes_[p_lgbm.argmax(1)], 'HOLD')
    res_lgbm   = simulate_trading(sig_lgbm, fwd_test)
    lgbm_fold_results.append(res_lgbm)

    # ── LR ────────────────────────────────────────────────────────────────────
    lr_model = make_lr_pipeline(C=best_lr_C, l1_ratio=best_lr_l1)
    lr_model.fit(X_train, y_train)
    p_lr     = lr_model.predict_proba(X_test)
    max_p_lr = p_lr.max(axis=1)
    cls_lr   = lr_model.classes_

    # threshold sweep
    for t in thresholds:
        sig = np.where(max_p_lr >= t, cls_lr[p_lr.argmax(1)], 'HOLD')
        lr_thresh_results[t].append(simulate_trading(sig, fwd_test))

    sig_lr   = np.where(max_p_lr >= PROBA_THRESHOLD,
                        cls_lr[p_lr.argmax(1)], 'HOLD')
    res_lr   = simulate_trading(sig_lr, fwd_test)
    lr_fold_results.append(res_lr)

    # ── Buy & Hold ────────────────────────────────────────────────────────────
    bh_curve = np.cumprod(1 + fwd_test)
    bh_ret   = bh_curve[-1] - 1
    bh_sh    = fwd_test.mean() / (fwd_test.std() + 1e-9) * np.sqrt(252)
    bh_dd    = ((bh_curve - np.maximum.accumulate(bh_curve))
                / np.maximum.accumulate(bh_curve)).min()
    bh_fold_results.append({'total_return': bh_ret, 'sharpe': bh_sh, 'max_drawdown': bh_dd})

    print(f'  {fold:<4} {str(d0)+" → "+str(d1):<26}'
          f'  {res_lgbm["total_return"]:>+10.2%} {res_lgbm["sharpe"]:>8.2f}'
          f'  {res_lr["total_return"]:>+10.2%} {res_lr["sharpe"]:>8.2f}'
          f'  (B&H {bh_ret:>+.0%})')

def avg(lst, k): return np.mean([r[k] for r in lst])

print('  ' + '-'*88)
print(f'  {"AVG":<31}'
      f'  {avg(lgbm_fold_results,"total_return"):>+10.2%} {avg(lgbm_fold_results,"sharpe"):>8.2f}'
      f'  {avg(lr_fold_results,"total_return"):>+10.2%} {avg(lr_fold_results,"sharpe"):>8.2f}'
      f'  (B&H {avg(bh_fold_results,"total_return"):>+.0%})')


In [ ]:
# Visualise the backtest results — 4 charts
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Production Backtest — LGBM vs Logistic Regression',
             fontsize=13, fontweight='bold')

# Collect all test dates
test_dates = []
for _, test_idx in splits:
    test_dates.extend(df.iloc[test_idx]['date'].values)
test_dates = pd.to_datetime(test_dates)

# Chain fold equity curves into one continuous curve
def chain_equity(fold_list):
    equity = [1.0]
    for res in fold_list:
        scale = equity[-1] / res['equity'][0] if res['equity'][0] != 0 else 1
        equity.extend((res['equity'] * scale).tolist())
    return np.array(equity[1:])

lgbm_equity = chain_equity(lgbm_fold_results)
lr_equity   = chain_equity(lr_fold_results)
bh_equity   = np.cumprod(1 + fwd_returns[splits[0][1][0]:])[:len(lgbm_equity)]

# Chart 1: Equity curves
ax = axes[0, 0]
ax.plot(test_dates[:len(lgbm_equity)], lgbm_equity, color='#185FA5', linewidth=1.5, label='LGBM')
ax.plot(test_dates[:len(lr_equity)],   lr_equity,   color='#639922', linewidth=1.5, label='LR', linestyle='--')
ax.axhline(1.0, color='black', linewidth=0.4, linestyle=':')
import matplotlib.ticker as mtick
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Equity curves (chained folds)', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=8)

# Chart 2: Log-loss comparison — full journey
ax = axes[0, 1]
model_labels = ['Random\nbaseline', 'LR\nbaseline', 'LR\ntuned', 'LGBM\nbaseline', 'LGBM\ntuned']
ll_values    = [np.log(3), avg_lr_base_ll, avg_lr_tuned_ll, avg_baseline_ll, avg_tuned_ll]
bar_colors   = ['#888888', '#185FA5', '#0d3d6b', '#639922', '#3B6D11']
bars = ax.bar(model_labels, ll_values, color=bar_colors, alpha=0.9, width=0.55)
for bar, val in zip(bars, ll_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold', color='#333')
ax.set_ylim(0.65, 1.22)
ax.set_title('Log-loss — full journey (lower = better)', fontsize=11)
ax.set_ylabel('Log-loss'); ax.tick_params(labelsize=8)

# Chart 3: Per-fold Sharpe comparison
ax = axes[1, 0]
fold_labels  = [f'F{i+1}' for i in range(N_SPLITS)]
lgbm_sharpes = [r['sharpe'] for r in lgbm_fold_results]
lr_sharpes   = [r['sharpe'] for r in lr_fold_results]
bh_sharpes   = [b['sharpe'] for b in bh_fold_results]
x = np.arange(N_SPLITS); w = 0.27
ax.bar(x - w, lgbm_sharpes, w, label='LGBM', color='#185FA5', alpha=0.85)
ax.bar(x,     lr_sharpes,   w, label='LR',   color='#639922', alpha=0.85)
ax.bar(x + w, bh_sharpes,   w, label='B&H',  color='#888780', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(fold_labels, fontsize=9)
ax.set_title('Sharpe ratio per fold', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=8)

# Chart 4: LR threshold sweep
ax = axes[1, 1]
thresh_returns = [avg(lr_thresh_results[t], 'total_return') for t in thresholds]
thresh_sharpes = [avg(lr_thresh_results[t], 'sharpe') for t in thresholds]
x2 = np.arange(len(thresholds)); w2 = 0.35
ax.bar(x2 - w2/2, thresh_returns, w2, label='Avg Return', color='#185FA5', alpha=0.85)
ax.bar(x2 + w2/2, thresh_sharpes, w2, label='Avg Sharpe', color='#639922', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xticks(x2); ax.set_xticklabels([f'LR thresh {t}' for t in thresholds], fontsize=9)
ax.set_title('LR threshold sweep', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as model_comparison.png')


---
## 14. Head-to-Head Comparison & Final Summary

Time to put all the numbers side by side and draw clear conclusions.

### Key concepts recap before reading the results

**Log-loss** — how confident and correct the model is. Lower is better. Random guessing = 1.0986.

**Sharpe ratio** — return divided by risk. Higher is better.
- Below 0 = lost money
- 0 to 1 = mediocre
- Above 1 = good (professional standard)

**Max drawdown** — the worst drop from peak to trough. Smaller (less negative) is better.

**Why these two models behave so differently:**  
LightGBM builds hundreds of decision trees that capture complex curved patterns. Logistic Regression draws one straight line. Bitcoin's market behaviour is highly non-linear — volatility regimes, momentum, fear cycles — so the curved boundary wins decisively here. This is not a universal truth: on simpler, more linear datasets, Logistic Regression would compete well.


In [ ]:
# Final summary table
print('=' * 72)
print(f'  {"FINAL HEAD-TO-HEAD COMPARISON":^68}')
print('=' * 72)

rand_ll = np.log(3)

rows = [
    ('Metric',             'Random',        'LR baseline', 'LR tuned',       'LGBM baseline', 'LGBM tuned ✅'),
    ('-'*22,               '-'*10,          '-'*13,        '-'*13,           '-'*13,          '-'*13),
    ('Log-loss',           f'{rand_ll:.4f}',f'{avg_lr_base_ll:.4f}',
                                                           f'{avg_lr_tuned_ll:.4f}',
                                                                              f'{avg_baseline_ll:.4f}',
                                                                                               f'{avg_tuned_ll:.4f}'),
    ('Avg return/fold',    '—',
     f'{avg(lr_base_res_per_fold,"total_return"):>+.1%}',
     f'{avg(lr_tuned_res_per_fold,"total_return"):>+.1%}',
     '—',
     f'{avg(lgbm_fold_results,"total_return"):>+.1%}'),
    ('Avg Sharpe',         '—',
     f'{avg(lr_base_res_per_fold,"sharpe"):.2f}',
     f'{avg(lr_tuned_res_per_fold,"sharpe"):.2f}',
     '—',
     f'{avg(lgbm_fold_results,"sharpe"):.2f}'),
    ('Avg max drawdown',   '—',
     f'{avg(lr_base_res_per_fold,"max_drawdown"):.1%}',
     f'{avg(lr_tuned_res_per_fold,"max_drawdown"):.1%}',
     '—',
     f'{avg(lgbm_fold_results,"max_drawdown"):.1%}'),
    ('Avg trades/fold',    '—',
     f'{avg(lr_base_res_per_fold,"n_trades"):.0f}',
     f'{avg(lr_tuned_res_per_fold,"n_trades"):.0f}',
     '—',
     f'{avg(lgbm_fold_results,"n_trades"):.0f}'),
]

for row in rows:
    print(f'  {row[0]:<22}  {row[1]:<12} {row[2]:<14} {row[3]:<14} {row[4]:<14} {row[5]}')

print()
print('=' * 72)
print(f'  {"WHY LGBM WINS":^68}')
print('=' * 72)
print("""
  LightGBM outperforms Logistic Regression on every metric because:

  1. Non-linearity — The boundary between BUY/HOLD/SELL days in
     28-dimensional feature space is NOT a straight line. Bitcoin's
     price dynamics involve regime changes, volatility clustering,
     and momentum effects that require curved decision boundaries.
     LightGBM's trees can approximate these. LR cannot.

  2. Feature interactions — LR treats each feature independently.
     It cannot learn "high RSI AND rising volume AND widening bands
     together signal a breakout". Trees capture these combinations.

  3. Probability spread — LR produces overconfident, bunched
     probabilities near the class prior (0.76 HOLD). This makes
     thresholding ineffective. LGBM spreads probabilities more
     across the full [0,1] range.
""")

print('=' * 72)
print(f'  {"WHAT LR CONTRIBUTES":^68}')
print('=' * 72)
print("""
  Even though LR loses on performance, it provides something
  LightGBM cannot: INTERPRETABILITY.

  The coefficient table tells us:
  ✅  BUY signal: driven by Bollinger Band breakout conditions
                 (bb_width_20 ↑, bb_position_20 ↑)
  ✅  SELL signal: driven by VIX spike and large upper wick
                  (vix_level_ma7 ↑, upper_wick ↑)
  ✅  HOLD signal: when vol_ratio_7_30 is elevated — no clear regime
  ✅  L1 zeroed out 58% of features — most are linear noise

  This is economically meaningful and can guide future feature
  engineering for the LGBM model.
""")

print('=' * 72)
print(f'  {"RECOMMENDED NEXT STEPS":^68}')
print('=' * 72)
print("""
  1. Temperature scaling   — improve LGBM probability calibration
     without flattening the signal spread (unlike sigmoid cal)

  2. Margin-based signals  — fire on P(BUY) - P(HOLD) > threshold
     instead of max_prob > threshold

  3. Add new features      — funding rate, open interest, on-chain
     flows (biggest potential ceiling lift)

  4. Ensemble             — combine LGBM + LR predictions
     (LGBM for performance, LR for calibration)
""")


In [ ]:
# Save both final models
import joblib

# Train on ALL data for deployment
lgbm_final = lgb.LGBMClassifier(**TUNED_PARAMS)
lgbm_final.fit(X, y)

lr_final_deploy = make_lr_pipeline(C=best_lr_C, l1_ratio=best_lr_l1)
lr_final_deploy.fit(X, y)

joblib.dump(lgbm_final,       'btc_lgbm_final.pkl')
joblib.dump(lr_final_deploy,  'btc_lr_final.pkl')

print('Models saved:')
print('  btc_lgbm_final.pkl  ← use this for live trading signals')
print('  btc_lr_final.pkl    ← use this for feature interpretation')
print()
print('To use them:')
print("  import joblib, numpy as np")
print("  lgbm = joblib.load('btc_lgbm_final.pkl')")
print("  lr   = joblib.load('btc_lr_final.pkl')")
print("  ")
print("  # Prepare today's features as a row (same 28 columns, same order)")
print("  proba    = lgbm.predict_proba(X_today)")
print("  signal   = lgbm.classes_[proba.argmax()] if proba.max() >= 0.45 else 'HOLD'")
print("  print(f'Signal: {signal}  (confidence: {proba.max():.1%})')")


---
## 15. All-Model Comparative Results — Full Summary

We bring together **all 8 trained models** (3 LGBM variants + 1 LGBM tuned + 3 LR variants + 1 LR tuned) into a single unified comparison.

Metrics reported:
- **Log-loss** — primary quality metric (lower = better; random = 1.099)
- **Average return per fold** — strategy performance
- **Average Sharpe ratio** — risk-adjusted return


In [ ]:
# ── Unified results table for all 8 models ────────────────────────────────────

all_model_summaries = {}

# LGBM variants
for vname in LGBM_VARIANT_CONFIGS:
    all_model_summaries[vname] = {
        'type'      : 'LightGBM',
        'avg_ll'    : np.mean(lgbm_variant_fold_ll[vname]),
        'avg_ret'   : np.mean(lgbm_variant_fold_ret[vname]),
        'avg_sharpe': np.mean(lgbm_variant_fold_sh[vname]),
    }

# LGBM Tuned (from sections 8-13)
all_model_summaries['LGBM-V4 (Tuned)'] = {
    'type'      : 'LightGBM',
    'avg_ll'    : np.mean(tuned_ll_per_fold),
    'avg_ret'   : np.mean([r['total_return'] for r in lgbm_fold_results]),
    'avg_sharpe': np.mean([r['sharpe']       for r in lgbm_fold_results]),
}

# LR variants
for vname in LR_VARIANT_CONFIGS:
    all_model_summaries[vname] = {
        'type'      : 'LogReg',
        'avg_ll'    : np.mean(lr_variant_fold_ll[vname]),
        'avg_ret'   : np.mean(lr_variant_fold_ret[vname]),
        'avg_sharpe': np.mean(lr_variant_fold_sh[vname]),
    }

# LR Tuned (from section 12)
all_model_summaries['LR-V4 (Tuned)'] = {
    'type'      : 'LogReg',
    'avg_ll'    : avg_lr_tuned_ll,
    'avg_ret'   : np.mean([r['total_return'] for r in lr_tuned_res_per_fold]),
    'avg_sharpe': np.mean([r['sharpe']       for r in lr_tuned_res_per_fold]),
}

rand_ll = np.log(3)

print('=' * 82)
print(f"  {'ALL-MODEL COMPARISON':^78}")
print('=' * 82)
print(f"  {'Model':<28}  {'Type':>8}  {'Avg LL':>8}  {'vs Rnd':>8}  {'Avg Ret':>9}  {'Avg Sh':>8}")
print('  ' + '-'*80)

best_ll = min(s['avg_ll'] for s in all_model_summaries.values())
for mname, stats in sorted(all_model_summaries.items(), key=lambda x: x[1]['avg_ll']):
    marker = '  <- BEST' if stats['avg_ll'] == best_ll else ''
    print(f"  {mname:<28}  {stats['type']:>8}  {stats['avg_ll']:>8.4f}"
          f"  {rand_ll-stats['avg_ll']:>+8.4f}  {stats['avg_ret']:>+9.2%}"
          f"  {stats['avg_sharpe']:>8.2f}{marker}")

print('  ' + '-'*80)
print(f"  {'Random baseline':<28}  {'  --':>8}  {rand_ll:>8.4f}  {'0.0000':>8}  {'--':>9}  {'--':>8}")


In [ ]:
# ── Visualise all-model comparison ────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('All-Model Comparative Results - 8 Models Side-by-Side',
             fontsize=13, fontweight='bold')

mnames    = list(all_model_summaries.keys())
mtypes    = [all_model_summaries[m]['type']       for m in mnames]
lls       = [all_model_summaries[m]['avg_ll']     for m in mnames]
rets      = [all_model_summaries[m]['avg_ret']    for m in mnames]
sharpes   = [all_model_summaries[m]['avg_sharpe'] for m in mnames]
bcolors   = ['#185FA5' if t=='LightGBM' else '#639922' for t in mtypes]
xlabels   = [m.replace(' (', '\n(') for m in mnames]
x         = np.arange(len(mnames))

# Chart 1: Log-loss
ax = axes[0,0]
bars = ax.bar(x, lls, color=bcolors, alpha=0.85, width=0.65)
ax.axhline(np.log(3), color='black', lw=1.2, ls='--', label='Random baseline')
for bar, val in zip(bars, lls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
            f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')
ax.set_title('Average Log-Loss per Model (lower=better)', fontsize=11)
ax.set_ylabel('Log-loss')
ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=7, rotation=30, ha='right')
ax.set_ylim(0.45, 1.25)
from matplotlib.patches import Patch
leg_els = [Patch(facecolor='#185FA5', label='LightGBM'),
           Patch(facecolor='#639922', label='LogReg'),
           plt.Line2D([0],[0], color='black', ls='--', label='Random')]
ax.legend(handles=leg_els, fontsize=8, loc='upper right')
ax.tick_params(labelsize=8)

# Chart 2: Scatter frontier
ax = axes[0,1]
for mname, stats in all_model_summaries.items():
    col = '#185FA5' if stats['type']=='LightGBM' else '#639922'
    ax.scatter(stats['avg_ll'], stats['avg_ret'], c=col, s=90, zorder=3)
    short = mname.split('(')[0].strip()
    ax.annotate(short, (stats['avg_ll'], stats['avg_ret']),
                fontsize=7, xytext=(4,4), textcoords='offset points')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.axvline(np.log(3), color='black', lw=0.5, ls='--')
ax.set_xlabel('Avg Log-Loss (lower=better)'); ax.set_ylabel('Avg Return')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Log-Loss vs Return - Model Frontier', fontsize=11)
ax.tick_params(labelsize=9)

# Chart 3: Return bars
ax = axes[1,0]
bars = ax.bar(x, rets, color=bcolors, alpha=0.80, width=0.65)
ax.axhline(0, color='black', lw=0.5)
for bar, val in zip(bars, rets):
    offset = 0.003 if val >= 0 else -0.008
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+offset,
            f'{val:+.1%}', ha='center', fontsize=8)
ax.set_title('Average Return per Fold (higher=better)', fontsize=11)
ax.set_ylabel('Return')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=7, rotation=30, ha='right')
ax.tick_params(labelsize=8)

# Chart 4: Sharpe bars
ax = axes[1,1]
bars = ax.bar(x, sharpes, color=bcolors, alpha=0.80, width=0.65)
ax.axhline(0, color='black', lw=0.5)
for bar, val in zip(bars, sharpes):
    offset = 0.03 if val >= 0 else -0.12
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+offset,
            f'{val:.2f}', ha='center', fontsize=8)
ax.set_title('Average Sharpe Ratio per Fold (higher=better)', fontsize=11)
ax.set_ylabel('Sharpe Ratio')
ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=7, rotation=30, ha='right')
ax.tick_params(labelsize=8)

plt.tight_layout(pad=2.0)
plt.savefig('all_models_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as all_models_comparison.png')


---
## 16. Data Split Ratio Investigation

We now investigate how the **train / validation / test split ratio** affects the final selected model (LGBM Tuned).  
Two ratios are tested on sequential (time-ordered) splits:

| Split | Train | Validation | Test | Rationale |
|---|---|---|---|---|
| **70 / 15 / 15** | 70% | 15% | 15% | Maximises training data; smaller holdout |
| **60 / 20 / 20** | 60% | 20% | 20% | Larger holdout for more reliable test estimates |

**Key rule:** splits must be sequential — never shuffle time-series data.  
Validation checks for overfitting; the test set is the final, unseen evaluation.


In [ ]:
# ── Fixed train / validation / test splits on LGBM Tuned ─────────────────────

SPLIT_RATIOS = {
    '70-15-15': (0.70, 0.15, 0.15),
    '60-20-20': (0.60, 0.20, 0.20),
}

n_total      = len(X)
split_results= {}

print('Data Split Ratio Investigation — Best Model: LGBM Tuned\n')

for split_name, (tr_f, va_f, te_f) in SPLIT_RATIOS.items():
    tr_end = int(n_total * tr_f)
    va_end = int(n_total * (tr_f + va_f))

    X_tr, y_tr   = X[:tr_end],       y[:tr_end]
    X_va, y_va   = X[tr_end:va_end], y[tr_end:va_end]
    X_te, y_te   = X[va_end:],       y[va_end:]
    fwd_va = fwd_returns[tr_end:va_end]
    fwd_te = fwd_returns[va_end:]

    model = lgb.LGBMClassifier(**TUNED_PARAMS)
    model.fit(X_tr, y_tr)

    def eval_set(Xs, ys, fwd):
        p   = model.predict_proba(Xs)
        ll  = log_loss(ys, p, labels=CLASS_NAMES)
        sig = np.where(p.max(1) >= PROBA_THRESHOLD,
                       model.classes_[p.argmax(1)], 'HOLD')
        res = simulate_trading(sig, fwd)
        return ll, res

    ll_va, res_va = eval_set(X_va, y_va, fwd_va)
    ll_te, res_te = eval_set(X_te, y_te, fwd_te)

    split_results[split_name] = {
        'tr_n': tr_end, 'va_n': va_end-tr_end, 'te_n': n_total-va_end,
        'll_va': ll_va, 'll_te': ll_te,
        'ret_va': res_va['total_return'], 'ret_te': res_te['total_return'],
        'sh_va' : res_va['sharpe'],       'sh_te' : res_te['sharpe'],
        'eq_va' : res_va['equity'],       'eq_te' : res_te['equity'],
        'dates_va'  : df['date'].iloc[tr_end:va_end].values,
        'dates_te'  : df['date'].iloc[va_end:].values,
        'd_tr_start': df['date'].iloc[0].date(),
        'd_tr_end'  : df['date'].iloc[tr_end-1].date(),
        'd_va_end'  : df['date'].iloc[va_end-1].date(),
        'd_te_end'  : df['date'].iloc[-1].date(),
    }

    sr = split_results[split_name]
    print(f'Split {split_name}:')
    print(f'  Train : {sr["d_tr_start"]} -> {sr["d_tr_end"]}  ({sr["tr_n"]} days)')
    print(f'  Val   : {sr["d_tr_end"]}  -> {sr["d_va_end"]}  ({sr["va_n"]} days)')
    print(f'  Test  : {sr["d_va_end"]}  -> {sr["d_te_end"]}  ({sr["te_n"]} days)')
    print(f'  Validation  -- LL: {ll_va:.4f}  Ret: {res_va["total_return"]:+.2%}  Sharpe: {res_va["sharpe"]:.2f}')
    print(f'  Test        -- LL: {ll_te:.4f}  Ret: {res_te["total_return"]:+.2%}  Sharpe: {res_te["sharpe"]:.2f}')
    print()


In [ ]:
# ── Visualise split ratio effects ─────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Data Split Ratio Investigation - LGBM Tuned on Two Split Ratios',
             fontsize=13, fontweight='bold')

split_names = list(split_results.keys())
sp_colors   = ['#185FA5', '#639922']

# Chart 1: Equity curves - validation
ax = axes[0,0]
for sn, col in zip(split_names, sp_colors):
    sr = split_results[sn]
    ax.plot(pd.to_datetime(sr['dates_va']), sr['eq_va'],
            color=col, lw=1.8, label=f'{sn} - Validation')
ax.axhline(1.0, color='black', lw=0.4, ls=':')
ax.set_title('Equity Curve - Validation Set', fontsize=11)
ax.set_ylabel('Portfolio value'); ax.legend(fontsize=9)
ax.tick_params(labelsize=8)

# Chart 2: Equity curves - test
ax = axes[0,1]
for sn, col in zip(split_names, sp_colors):
    sr = split_results[sn]
    ax.plot(pd.to_datetime(sr['dates_te']), sr['eq_te'],
            color=col, lw=1.8, label=f'{sn} - Test (holdout)')
ax.axhline(1.0, color='black', lw=0.4, ls=':')
ax.set_title('Equity Curve - Test Holdout Set', fontsize=11)
ax.set_ylabel('Portfolio value'); ax.legend(fontsize=9)
ax.tick_params(labelsize=8)

# Chart 3: Key metrics bar chart
ax = axes[1,0]
metric_labels = ['LL Val', 'LL Test', 'Ret Val', 'Ret Test']
x4 = np.arange(len(metric_labels)); bw = 0.3
for i, (sn, col) in enumerate(zip(split_names, sp_colors)):
    sr   = split_results[sn]
    vals = [sr['ll_va'], sr['ll_te'], sr['ret_va'], sr['ret_te']]
    ax.bar(x4 + (i-0.5)*bw, vals, bw, color=col, alpha=0.8, label=sn)
ax.axhline(np.log(3), color='gray', lw=1, ls='--', label='Random LL')
ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x4); ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_title('Key Metrics: Val vs Test across Split Ratios', fontsize=11)
ax.legend(fontsize=8); ax.tick_params(labelsize=9)

# Chart 4: Sharpe ratio comparison
ax = axes[1,1]
x5 = np.arange(2); bw2 = 0.3
for i, (sn, col) in enumerate(zip(split_names, sp_colors)):
    sr = split_results[sn]
    ax.bar(x5 + (i-0.5)*bw2, [sr['sh_va'], sr['sh_te']],
           bw2, color=col, alpha=0.8, label=sn)
ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x5); ax.set_xticklabels(['Validation', 'Test (holdout)'], fontsize=10)
ax.set_title('Sharpe Ratio: Val vs Test', fontsize=11)
ax.set_ylabel('Sharpe Ratio'); ax.legend(fontsize=9)
ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('split_ratio_investigation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved as split_ratio_investigation.png')
print()
print('Key takeaway:')
for sn in split_names:
    sr  = split_results[sn]
    gap = abs(sr['ll_va'] - sr['ll_te'])
    verdict = 'Well-calibrated' if gap < 0.05 else 'Some generalisation gap'
    print(f'  {sn}: LL gap (val vs test) = {gap:.4f}  -> {verdict}')


---
## 17. Interactive Prediction Interface

Enter feature values below and click **Predict Signal** to receive a live prediction from the trained LGBM model.

**How to use this interface:**
1. Open `sample_inputs.txt` (provided alongside this notebook)
2. Copy one of the sample lines (the 28 comma-separated numbers — not the comment lines starting with `#`)
3. Paste into the text box below
4. Click **Predict Signal**

**Output explained:**
- **Signal**: BUY / HOLD / SELL — what the model would have recommended
- **Confidence**: the model's maximum class probability (must exceed 45% threshold to act)
- **Class Probabilities**: full breakdown across all three outcomes
- **Market Context**: plain-English interpretation of the key input features


In [ ]:
# ── Interactive Prediction Interface ─────────────────────────────────────────
# Paste a comma-separated row from sample_inputs.txt into the text box.
# Feature order matches the feature_cols list defined in Section 5.

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Re-train the final LGBM model on all data (if not already trained) ────────
lgbm_interface_model = lgb.LGBMClassifier(**TUNED_PARAMS)
lgbm_interface_model.fit(X, y)

# ── Widget layout ─────────────────────────────────────────────────────────────
style       = {'description_width': '160px'}

header_html = widgets.HTML(value=(
    '<div style="background:#185FA5;color:white;padding:12px 18px;'
    'border-radius:6px;margin-bottom:10px;">'
    '<b style="font-size:16px;">BTC Direction Classifier - Live Prediction Interface</b><br>'
    '<span style="font-size:12px;">Paste a comma-separated feature row from sample_inputs.txt below</span>'
    '</div>'
))

input_box = widgets.Textarea(
    value='',
    placeholder='Paste comma-separated values here (28 numbers)\ne.g. 0.025, 0.018, -0.003, ...',
    description='Feature values:',
    layout=widgets.Layout(width='95%', height='80px'),
    style=style,
)

predict_btn = widgets.Button(
    description='Predict Signal',
    button_style='primary',
    layout=widgets.Layout(width='180px', height='38px'),
)

clear_btn = widgets.Button(
    description='Clear',
    button_style='',
    layout=widgets.Layout(width='100px', height='38px'),
)

output_area          = widgets.Output()
feature_count_label  = widgets.HTML(value='<span style="color:gray;font-size:11px;">No input yet</span>')

def parse_input(raw_text):
    parts = [p.strip() for p in raw_text.replace('\n', ',').split(',')]
    parts = [p for p in parts if p]
    return np.array([float(p) for p in parts])

def make_signal_html(signal, proba_dict, input_arr):
    sig_colors = {'BUY': '#2E7D32', 'SELL': '#B71C1C', 'HOLD': '#1565C0'}
    sig_icons  = {'BUY': 'UP', 'SELL': 'DOWN', 'HOLD': 'HOLD'}
    sig_bg     = {'BUY': '#E8F5E9', 'SELL': '#FFEBEE', 'HOLD': '#E3F2FD'}
    col        = sig_colors.get(signal, '#333')
    icon       = sig_icons.get(signal, '?')
    bg         = sig_bg.get(signal, '#fafafa')

    prob_bars = ''
    for cls in ['BUY', 'HOLD', 'SELL']:
        p   = proba_dict.get(cls, 0)
        bc  = sig_colors[cls]
        bar = ('<div style="display:inline-block;width:' + str(int(p*200)) +
               'px;height:14px;background:' + bc + ';opacity:0.7;border-radius:2px;"></div>')
        prob_bars += ('<div style="margin:3px 0;font-size:12px;">'
                      + cls + ': ' + f'{p:.1%}' + ' ' + bar + '</div>')

    feat_vals = dict(zip(feature_cols, input_arr))
    notes = []
    if feat_vals.get('rsi_14', 50) > 65:
        notes.append('RSI >65 - overbought territory')
    elif feat_vals.get('rsi_14', 50) < 35:
        notes.append('RSI <35 - oversold territory')
    if feat_vals.get('vix_level_ma7', 20) > 30:
        notes.append('VIX elevated - fearful market conditions')
    if feat_vals.get('fear_greed_ma7', 50) > 70:
        notes.append('Fear & Greed >70 - greed phase')
    elif feat_vals.get('fear_greed_ma7', 50) < 30:
        notes.append('Fear & Greed <30 - fear phase')
    if feat_vals.get('bb_position_20', 0.5) > 0.85:
        notes.append('Near upper Bollinger Band - price stretched')
    elif feat_vals.get('bb_position_20', 0.5) < 0.15:
        notes.append('Near lower Bollinger Band - price compressed')
    if not notes:
        notes.append('No extreme conditions detected - mixed signals')

    notes_html = ''.join('<li style="font-size:12px;">' + n + '</li>' for n in notes)
    conf       = max(proba_dict.values())

    html = (
        '<div style="background:' + bg + ';border-left:5px solid ' + col +
        ';padding:14px 18px;border-radius:6px;margin-top:12px;font-family:sans-serif;">'
        '<div style="font-size:22px;font-weight:bold;color:' + col + ';">' + icon + '  ' + signal + '</div>'
        '<div style="font-size:12px;color:#555;margin-top:4px;">'
        'Model confidence: <b>' + f'{conf:.1%}' + '</b> &nbsp;|&nbsp; '
        'Threshold: ' + f'{PROBA_THRESHOLD:.0%}' +
        '</div>'
        '<hr style="border:0;border-top:1px solid #ddd;margin:10px 0;">'
        '<b style="font-size:12px;">Class Probabilities:</b><br>' + prob_bars +
        '<hr style="border:0;border-top:1px solid #ddd;margin:10px 0;">'
        '<b style="font-size:12px;">Market Context (from input features):</b>'
        '<ul style="margin:4px 0 0 0;padding-left:18px;">' + notes_html + '</ul>'
        '</div>'
    )
    return html

def on_predict(btn):
    with output_area:
        clear_output()
        raw = input_box.value.strip()
        if not raw:
            display(HTML('<span style="color:red;">Please paste feature values first.</span>'))
            return
        try:
            arr    = parse_input(raw)
            n_feat = len(feature_cols)
            if len(arr) != n_feat:
                display(HTML('<span style="color:red;">Expected ' + str(n_feat) +
                             ' values, got ' + str(len(arr)) + '. Check your input.</span>'))
                return
            X_input    = arr.reshape(1, -1)
            proba      = lgbm_interface_model.predict_proba(X_input)[0]
            classes    = lgbm_interface_model.classes_
            proba_dict = dict(zip(classes, proba))
            signal     = classes[proba.argmax()] if max(proba) >= PROBA_THRESHOLD else 'HOLD'
            display(HTML(make_signal_html(signal, proba_dict, arr)))
        except ValueError as e:
            display(HTML('<span style="color:red;">Parse error: ' + str(e) + '</span>'))
        except Exception as e:
            display(HTML('<span style="color:red;">Error: ' + str(e) + '</span>'))

def on_clear(btn):
    input_box.value = ''
    with output_area:
        clear_output()
    feature_count_label.value = '<span style="color:gray;font-size:11px;">No input yet</span>'

def on_input_change(change):
    raw = change['new'].strip()
    if not raw:
        feature_count_label.value = '<span style="color:gray;font-size:11px;">No input yet</span>'
        return
    try:
        arr = parse_input(raw)
        n   = len(arr)
        exp = len(feature_cols)
        col = 'green' if n == exp else 'orange'
        msg = (str(n) + '/' + str(exp) + ' values parsed - ready!' if n == exp
               else str(n) + '/' + str(exp) + ' values - need exactly ' + str(exp))
        feature_count_label.value = '<span style="color:' + col + ';font-size:11px;">' + msg + '</span>'
    except Exception:
        feature_count_label.value = '<span style="color:red;font-size:11px;">Cannot parse input</span>'

predict_btn.on_click(on_predict)
clear_btn.on_click(on_clear)
input_box.observe(on_input_change, names='value')

feat_items = ' | '.join(str(i+1) + '.' + c for i, c in enumerate(feature_cols))
feature_ref = widgets.HTML(value=(
    '<div style="background:#f5f5f5;padding:10px;border-radius:4px;margin-top:8px;">'
    '<b style="font-size:11px;">Feature order (' + str(len(feature_cols)) + ' values):</b><br>'
    '<span style="font-size:10px;color:#555;">' + feat_items + '</span>'
    '</div>'
))

display(header_html)
display(input_box)
display(feature_count_label)
display(widgets.HBox([predict_btn, clear_btn]))
display(output_area)
display(feature_ref)
